In [ ]:
# %pip install llama-cpp-python huggingface_hub

In [ ]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama
import os

In [ ]:
os.environ['HF_HUB_DISABLE_SSL_VERIFY'] = '1'

In [ ]:
model_path = os.getcwd() + ("\\Qwen3-0.6B-Q2_K.gguf")
print(model_path)

In [ ]:
llm = Llama(model_path=model_path, n_ctx=2048, verbose=False)

In [ ]:
# Create a dummy file for testing if it doesn't exist
with open("test_diff.txt", "w") as f:
    f.write("""diff --git a/src/main.rs b/src/main.rs
index e69de29..8b13789 100644
--- a/src/main.rs
+++ b/src/main.rs
@@ -1,1 +1,5 @@
-fn main() { println!("hello"); }
+fn main() {
+    let args: Vec<String> = std::env::args().collect();
+    let config = Config::new(&args);
+    println!("Running with config: {:?}", config);
+}""")

In [ ]:
def get_git_diff(file_path):
    with open(file_path, "r") as f:
        return f.read()

git_diff = get_git_diff("test_diff.txt")
print("Diff loaded successfully.")

In [ ]:
prompt = f"""<|im_start|>system
You are a concise git commit generator. Output ONLY the message. No thinking.<|im_end|>
<|im_start|>user
{git_diff}<|im_end|>
<|im_start|>assistant
<think>
Done analyzing.</think>
feat:""" 
# Pre-filling "feat:" forces the model to skip thinking and start the message

In [ ]:
output = llm(
    prompt,
    max_tokens=100,
    stop=["<|im_end|>"],
    temperature=0.1 # Low temperature for consistent, logical messages
)


In [ ]:
print("\n--- GENERATED COMMIT MESSAGE ---")
print(output["choices"][0]["text"].strip())